In [ ]:
import numpy as np
import numpy.random
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.utils import resample

Failed to import duecredit due to No module named 'duecredit'


In [2]:
intercept = 0.0
slope = 1.0
n = 11
x = np.linspace(0, 1.0, num=n)
sd_err = 0.1
seed = 1234
y = x + numpy.random.default_rng(seed=seed).normal(scale=sd_err, size=n)
x_pred = 0.75

I copy/pasted these values into R and did the linear regression there:

```
> predict(m, newdata=list(x=0.75), interval='confidence')
        fit       lwr       upr
1 0.7537439 0.6289261 0.8785618
```

The bootstrap estimate over linear regressions is close:


In [3]:
def lm(x, y, x_pred=x_pred):
    return (
        LinearRegression().fit(X=x.reshape(-1, 1), y=y).predict(np.array([[x_pred]]))[0]
    )


def bootstrap(f, x=x, y=y, **kwargs):
    bx, by = resample(x, y)
    return f(bx, by, **kwargs)


def summarize(est, alpha=0.05):
    print(
        f"{est.mean()} ({np.quantile(est, alpha / 2)} -- {np.quantile(est, 1 - alpha / 2)})"
    )


n_boot = 1000
summarize(np.array([bootstrap(lm) for _ in range(n_boot)]))

0.7558227669412172 (0.6666780987764388 -- 0.8639165675964497)


In [4]:
def rf(x, y, **kwargs):
    return (
        RandomForestRegressor(**kwargs)
        .fit(X=x.reshape(-1, 1), y=y)
        .predict(np.array([[x_pred]]))[0]
    )


print("Bootstrap interval over individual trees:")
summarize(np.array([bootstrap(rf, n_estimators=1) for _ in range(1000)]))
n_trees = 100
print(f"Over forests of size {n_trees}:")
summarize(np.array([bootstrap(rf, n_estimators=n_trees) for _ in range(1000)]))

Bootstrap interval over individual trees:
0.7149150752230818 (0.45211766393355984 -- 0.934374458145268)
Over forests of size 100:
0.7139439629786212 (0.4962020473506204 -- 0.8467003695349685)


In [9]:
alpha = 0.05
for a in [alpha / 2, 0.5, 1 - alpha / 2]:
    pred = (
        GradientBoostingRegressor(loss="quantile", alpha=a, n_estimators=100)
        .fit(X=x.reshape(-1, 1), y=y)
        .predict(np.array([[x_pred]]))
    )
    print(f"{a=} {pred=}")

a=0.025 pred=array([0.10640468])
a=0.5 pred=array([0.64646172])
a=0.975 pred=array([0.93437474])
